# Ch.2 — Multiple Linear Regression

**Goal**: Use all 8 California Housing features to improve MAE from $70k → $55k.

| What | Value |
|------|-------|
| Dataset | California Housing (20,640 districts, 8 features) |
| Ch.1 baseline | ~$70k MAE (single feature: MedInc) |
| This chapter target | ~$55k MAE (all 8 features) |
| Grand Challenge target | <$40k MAE |

## Setup & Data Loading

In [ ]:
# TODO: Implement this cell
#  (Imports)
#
# Steps:
# 1. Import the required libraries: matplotlib.pyplot, numpy, pandas, seaborn, sklearn.datasets, sklearn.linear_model
# 2. Set random seed (np.random.seed) and configure the plot style
#
# Hint:
#   import numpy as np
#   import pandas as pd
#   import matplotlib.pyplot as plt

In [ ]:
# TODO: Implement this cell
#  (Load the California Housing dataset)
#
# Steps:
# 1. Call fetch_california_housing() to load the dataset object
# 2. Create a pandas DataFrame: pd.DataFrame(housing.data, columns=housing.feature_names)
# 3. Append target column: df['MedHouseVal'] = housing.target
# 4. Print dataset shape and preview with df.head()
#
# Hint:
#   housing = fetch_california_housing()
#   df = pd.DataFrame(housing.data, columns=housing.feature_names)
#   df['MedHouseVal'] = housing.target

## Feature Correlation Analysis

Before fitting, we need to understand how features relate to each other.
Highly correlated features cause **multicollinearity** — unstable individual weights.

In [ ]:
# TODO: Implement this cell
#  (Feature correlation heatmap)
#
# Steps:
# 1. Compute X.corr() for the feature correlation matrix
# 2. Create an upper-triangle mask: np.triu(np.ones_like(corr, dtype=bool))
# 3. Plot with sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r')
# 4. Print pairs where |ρ| > 0.7 as collinear
#
# Hint:
#   corr = X.corr()
#   mask = np.triu(np.ones_like(corr, dtype=bool))
#   sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0)

## VIF — Variance Inflation Factor

VIF quantifies how much each feature's coefficient variance is inflated by collinearity.
- VIF = 1: no collinearity
- VIF > 5: moderate concern
- VIF > 10: severe — drop or regularize

In [ ]:
# TODO: Implement this cell
#  (VIF computation)
#
# Steps:
# 1. Standardize X with StandardScaler (fit on train only)
# 2. Prepend a column of ones: X_vif = np.column_stack([np.ones(n), X_scaled])
# 3. For i, name in enumerate(feature_names): compute variance_inflation_factor(X_vif, i+1)
# 4. Print each feature with its VIF and flag rows where VIF > 5 as HIGH
#
# Hint:
#   from statsmodels.stats.outliers_influence import variance_inflation_factor
#   X_vif = np.column_stack([np.ones(X_scaled.shape[0]), X_scaled])
#   vif = variance_inflation_factor(X_vif, i + 1)   # i=0 means feature index 0

## Single-Feature Baseline (Ch.1 Recap)

First, establish the Ch.1 baseline so we can measure the improvement.

In [ ]:
# TODO: Implement this cell
#  (Ch.1 baseline: MedInc only)
#
# Steps:
# 1. Call train_test_split(X, y, test_size=0.2, random_state=SEED)
# 2. Print resulting train/test shapes
#
# Hint:
#   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=???, random_state=???)

## Multi-Feature Model — All 8 Features

Now use ALL features. The key question: how much does adding 7 more features help?

In [ ]:
# TODO: Implement this cell
#  (Multi-feature model: all 8 features)
#
# Steps:
# 1. Scale all 8 features with StandardScaler (fit on train only)
# 2. Fit LinearRegression on scaled train data
# 3. Compute MAE (×100_000) and R² on test data
# 4. Print improvement over the single-feature baseline
#
# Hint:
#   scaler_train = StandardScaler()
#   X_train_s = scaler_train.fit_transform(X_train)  # 8 features
#   X_test_s  = scaler_train.transform(X_test)
#   model_8feat = LinearRegression().fit(X_train_s, y_train)

## Feature Importance — Which Features Matter?

After standardization, absolute weight magnitude indicates feature importance.
This only works because all features are on the same scale (mean=0, std=1).

In [ ]:
# TODO: Implement this cell
#  (Feature importance (standardized weights))
#
# Steps:
# 1. Extract model.coef_ from the fitted model (standardized weights)
# 2. Create a DataFrame: {'Feature': names, 'Weight': coef_, 'Abs Weight': abs(coef_)} and sort by abs
# 3. Plot ax.barh with positive weights in green, negative in red
# 4. Annotate each bar with its weight value; add title and axis labels
# 5. Print top-N features by absolute weight
#
# Hint:
#   importance = pd.DataFrame({'Feature': feature_names, 'Weight': model.coef_})
#   importance['Abs Weight'] = importance['Weight'].abs()
#   importance = importance.sort_values('Abs Weight', ascending=True)
#   ax.barh(importance['Feature'], importance['Weight'], color=colors)

## Residual Analysis

The residual plot reveals whether the linear assumption holds.
- **Random scatter** → linear model is appropriate
- **U-shape or curve** → non-linear relationship (need polynomial features in Ch.3)

In [ ]:
# TODO: Implement this cell
#  (Residual plot)
#
# Steps:
# 1. Compute residuals = (y_test - y_pred) * 100_000
# 2. Create subplots: residuals-vs-predicted scatter (with axhline at 0) + residual histogram
# 3. Set titles, axis labels; call plt.tight_layout()
# 4. Save with plt.savefig and print residual summary statistics
#
# Hint:
#   residuals = (y_test - y_pred) * 100_000
#   fig, axes = plt.subplots(2, 2, figsize=(14, 12))
#   axes[0,0].scatter(y_test*100_000, y_pred*100_000, alpha=0.15, s=10)
#   axes[1,0].axhline(y=0, color='red', linewidth=2, linestyle='--')

## Manual Gradient Descent (Vectorized)

Educational implementation: same core loop as Ch.1, but now with 8 features.
The gradient formula generalizes naturally from scalar to vector.

In [ ]:
# TODO: Implement this cell
#  (Gradient descent from scratch (8 features))
#
# Steps:
# 1. Initialise W = np.zeros(X.shape[1]) and b = 0.0
# 2. Loop `epochs` times:
# 3.   a. Forward pass: y_hat = X @ W + b
# 4.   b. Gradients: dW = (2/n) * X.T @ (y_hat - y);  db = (2/n) * (y_hat-y).sum()
# 5.   c. Update: W -= alpha * dW;  b -= alpha * db
# 6.   d. Append mse(y, y_hat) to history
# 7. Return W, b, history
#
# Hint:
#   n, p = X.shape;  W = np.zeros(p);  b = 0.0
#   y_hat = X @ W + b
#   dW = (2/n) * X.T @ (y_hat - y);  db = (2/n) * (y_hat - y).sum()
#   W -= alpha * dW;  b -= alpha * db

## Comparison Summary

Side-by-side comparison of Ch.1 (single feature) vs Ch.2 (all features).

In [ ]:
# TODO: Implement this cell
#  (Summary comparison)
#
# Steps:
# 1. Summary comparison
#
# Hint:
#    # implement using the APIs described above

## Exercises

1. **Incremental features**: Add features one by one (MedInc → +HouseAge → +AveRooms → ...) and plot MAE vs number of features. Which feature gives the biggest single improvement?
2. **Drop collinear features**: Remove AveBedrms (correlated with AveRooms). Does MAE change? Does weight stability improve?
3. **Normal Equation from scratch**: Implement $\mathbf{w}^* = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ manually and compare to sklearn's result.

In [ ]:
# TODO: Implement this cell
#  (Exercise 1 scaffold: incremental feature addition)
#
# Steps:
# 1. Implement 'Exercise 1 scaffold: incremental feature addition' section
# 2. Implement the logic described in the surrounding markdown cell
#
# Hint:
#   # see solution cell for the required API calls

In [ ]:
# TODO: Implement this cell
#  (Exercise 2 scaffold: drop AveBedrms)
#
# Steps:
# 1. Exercise 2 scaffold: drop AveBedrms
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#  (Exercise 3 scaffold: Normal Equation from scratch)
#
# Steps:
# 1. Exercise 3 scaffold: Normal Equation from scratch
#
# Hint:
#    # implement using the APIs described above